In [9]:
################ Reading multiple .nc file ###################
import xarray as xr
from pathlib import Path

# Folder that contains your .nc files
data_dir = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")

# Loop over all NetCDF files in the folder
for nc_path in sorted(data_dir.glob("*.nc")):
    print("\n" + "=" * 80)
    print(f"File: {nc_path.name}")
    print("=" * 80)

    # Open the dataset
    with xr.open_dataset(nc_path) as ds:
        # Quick overview of the dataset
        print("\n--- xarray dataset summary ---")
        print(ds)

        # More explicit info on each data variable
        print("\n--- Variable details ---")
        for var_name, var in ds.data_vars.items():
            print(f"\nVariable name: {var_name}")
            print(f"  dims:  {var.dims}")
            print(f"  shape: {var.shape}")
            print(f"  attributes:")
            for k, v in var.attrs.items():
                print(f"    {k}: {v}")



File: ESI_dryz.nc

--- xarray dataset summary ---
<xarray.Dataset> Size: 826MB
Dimensions:   (time: 288, lat: 611, lon: 587)
Coordinates:
  * time      (time) datetime64[ns] 2kB 2000-01-31 2000-02-29 ... 2023-12-31
  * lat       (lat) float32 2kB 6.634 6.684 6.734 6.784 ... 37.03 37.08 37.13
  * lon       (lon) float32 2kB 68.26 68.31 68.36 68.41 ... 97.46 97.51 97.56
Data variables:
    ESI_dryz  (time, lat, lon) float64 826MB ...

--- Variable details ---

Variable name: ESI_dryz
  dims:  ('time', 'lat', 'lon')
  shape: (288, 611, 587)
  attributes:

File: SIF_dryz.nc

--- xarray dataset summary ---
<xarray.Dataset> Size: 413MB
Dimensions:   (time: 288, lat: 611, lon: 587)
Coordinates:
  * time      (time) datetime64[ns] 2kB 2000-01-31 2000-02-29 ... 2023-12-31
  * lat       (lat) float32 2kB 6.634 6.684 6.734 6.784 ... 37.03 37.08 37.13
  * lon       (lon) float32 2kB 68.26 68.31 68.36 68.41 ... 97.46 97.51 97.56
Data variables:
    SIF_dryz  (time, lat, lon) float32 413MB ...

---

In [19]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from pathlib import Path
import geopandas as gpd

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
data_dir = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
shp_path = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

datasets_info = [
    {"file": "ESI_dryz.nc",                             "var": "ESI_dryz",  "title": "ESI"},
    {"file": "SIF_dryz.nc",                             "var": "SIF_dryz",  "title": "SIF"},
    {"file": "SMroot_dryz.nc",                          "var": "SMroot_dryz", "title": "SMroot"},
    {"file": "TVDI_Z_monthly_India_0p05deg_2001-2023.nc","var": "TVDI_Z",   "title": "TVDI_Z"},
    {"file": "VOD_dryz.nc",                             "var": "VOD_dryz",  "title": "VOD"},
    {"file": "VDI_allbiomes.nc",                        "var": "VDI",       "title": "VDI"},
    {"file": "VDI_cropland.nc",                         "var": "VDI",       "title": "VDI_Cropland"},
    {"file": "VDI_forest.nc",                           "var": "VDI",       "title": "VDI_Forest"},
    {"file": "VDI_shrub_grass.nc",                      "var": "VDI",       "title": "VDI_Shrub_Grass"},
]

# ---------------------------------------------------------------------
# Load shapefile
# ---------------------------------------------------------------------
india_gdf = gpd.read_file(shp_path)

# Ensure shapefile is in lat/lon (EPSG:4326)
if india_gdf.crs is not None and india_gdf.crs.to_string() != "EPSG:4326":
    india_gdf = india_gdf.to_crs(epsg=4326)

# ---------------------------------------------------------------------
# Load and compute mean over time
# ---------------------------------------------------------------------
annual_means = []

for info in datasets_info:
    ds = xr.open_dataset(data_dir / info["file"])
    mean_da = ds[info["var"]].mean("time", skipna=True)
    annual_means.append(mean_da)
    ds.close()

# ---------------------------------------------------------------------
# Color settings
# ---------------------------------------------------------------------
vmin, vmax = -1, 1
norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

colors = [
    "#313695", "#4575b4", "#74add1", "#abd9e9",
    "#ffffff",
    "#fdae61", "#f46d43", "#d73027", "#a50026"
]
cmap = LinearSegmentedColormap.from_list("custom_div", colors)

# ---------------------------------------------------------------------
# Plot 3x3 panel WITHOUT CARTOPY
# ---------------------------------------------------------------------
fig, axes = plt.subplots(3, 3, figsize=(12,10))

im = None

for ax, mean_da, info in zip(axes.flat, annual_means, datasets_info):

    lats = mean_da["lat"].values
    lons = mean_da["lon"].values

    lon2d, lat2d = np.meshgrid(lons, lats)

    # PLOT THE VARIABLE
    im = ax.pcolormesh(
        lon2d, lat2d, mean_da.values,
        cmap=cmap,
        norm=norm,
        shading="auto"
    )

    # OVERLAY SHAPEFILE
    india_gdf.boundary.plot(ax=ax, color="black", linewidth=0.5)

    ax.set_title(info["title"], fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

# ---------------------------------------------------------------------
# One colorbar on right
# ---------------------------------------------------------------------
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Standardized anomaly (z-score)")

# ---------------------------------------------------------------------
# Save output
# ---------------------------------------------------------------------
out_file = data_dir / "annual_mean_3x3_panel_with_boundary.png"
fig.savefig(out_file, dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_file)


Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\annual_mean_3x3_panel_with_boundary.png


In [21]:
################ Seasonal Climatology ##############
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from pathlib import Path
import geopandas as gpd

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
data_dir = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
shp_path = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

# List of datasets / variables / names to use in titles & filenames
datasets_info = [
    {"file": "ESI_dryz.nc",                             "var": "ESI_dryz",  "title": "ESI"},
    {"file": "SIF_dryz.nc",                             "var": "SIF_dryz",  "title": "SIF"},
    {"file": "SMroot_dryz.nc",                          "var": "SMroot_dryz", "title": "SMroot"},
    {"file": "TVDI_Z_monthly_India_0p05deg_2001-2023.nc","var": "TVDI_Z",   "title": "TVDI_Z"},
    {"file": "VOD_dryz.nc",                             "var": "VOD_dryz",  "title": "VOD"},
    {"file": "VDI_allbiomes.nc",                        "var": "VDI",       "title": "VDI"},
    {"file": "VDI_cropland.nc",                         "var": "VDI",       "title": "VDI_Cropland"},
    {"file": "VDI_forest.nc",                           "var": "VDI",       "title": "VDI_Forest"},
    {"file": "VDI_shrub_grass.nc",                      "var": "VDI",       "title": "VDI_Shrub_Grass"},
]

# Seasons and their month definitions
season_order = ["DJF", "MAM", "JJAS", "ON"]
season_months = {
    "DJF":  [12, 1, 2],
    "MAM":  [3, 4, 5],
    "JJAS": [6, 7, 8, 9],
    "ON":   [10, 11],
}

# ---------------------------------------------------------------------
# Load shapefile
# ---------------------------------------------------------------------
india_gdf = gpd.read_file(shp_path)

# Ensure shapefile is in lat/lon
if india_gdf.crs is not None and india_gdf.crs.to_string() != "EPSG:4326":
    india_gdf = india_gdf.to_crs(epsg=4326)

# ---------------------------------------------------------------------
# Common colour settings: –2 to +2, 0 = white
# ---------------------------------------------------------------------
vmin, vmax = -2, 2
norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

colors = [
    "#313695", "#4575b4", "#74add1", "#abd9e9",
    "#ffffff",
    "#fdae61", "#f46d43", "#d73027", "#a50026"
]
cmap = LinearSegmentedColormap.from_list("custom_div", colors)

# ---------------------------------------------------------------------
# Helper: assign season labels based on month
# ---------------------------------------------------------------------
def month_to_season(month_arr):
    """Return array of season labels (DJF, MAM, JJAS, ON) for given month integers."""
    out = np.empty(month_arr.shape, dtype=object)
    for s, months in season_months.items():
        mask = np.isin(month_arr, months)
        out[mask] = s
    return out

# ---------------------------------------------------------------------
# Main loop: one figure per variable, 2x2 panel of seasons
# ---------------------------------------------------------------------
for info in datasets_info:
    print(f"Processing seasonal climatology for: {info['title']}")

    # Load dataset and variable
    ds = xr.open_dataset(data_dir / info["file"])
    da = ds[info["var"]]

    # Get month numbers from time coordinate
    months = da["time"].dt.month.values  # shape (time,)

    # Build season label array
    season_labels = month_to_season(months)

    # Wrap as DataArray so we can use groupby
    season_da = xr.DataArray(
        season_labels,
        coords={"time": da["time"]},
        dims=("time",),
        name="season"
    )

    # Group by season and average over time (climatology)
    # Result dims: season, lat, lon
    seasonal_clim = da.groupby(season_da).mean("time", skipna=True)

    ds.close()

    # -----------------------------------------------------------------
    # Plot 2x2 seasonal maps for this variable
    # -----------------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    axes = axes.ravel()

    im = None

    # Use lat/lon from the data
    lats = da["lat"].values
    lons = da["lon"].values
    lon2d, lat2d = np.meshgrid(lons, lats)

    for i, season in enumerate(season_order):
        ax = axes[i]

        # Skip season if not present in data (just in case)
        if season not in seasonal_clim["season"]:
            ax.set_visible(False)
            continue

        data_season = seasonal_clim.sel(season=season)

        im = ax.pcolormesh(
            lon2d,
            lat2d,
            data_season.values,
            cmap=cmap,
            norm=norm,
            shading="auto"
        )

        # Overlay India boundary
        india_gdf.boundary.plot(ax=ax, color="black", linewidth=0.5)

        ax.set_title(season, fontsize=11)
        ax.set_xticks([])
        ax.set_yticks([])

    # Turn off any unused axes (if any)
    for j in range(len(season_order), len(axes)):
        axes[j].set_visible(False)

    # Shared colorbar on the right
    fig.subplots_adjust(right=0.88)
    cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label("Standardized anomaly (z-score)")

    # Main title
    fig.suptitle(f"{info['title']} seasonal mean (DJF, MAM, JJAS, ON)", fontsize=14)

    # Save figure
    out_file = data_dir / f"{info['title']}_seasonal_DJF_MAM_JJAS_ON.png"
    fig.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"  Saved: {out_file}")


Processing seasonal climatology for: ESI
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\ESI_seasonal_DJF_MAM_JJAS_ON.png
Processing seasonal climatology for: SIF
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\SIF_seasonal_DJF_MAM_JJAS_ON.png
Processing seasonal climatology for: SMroot
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\SMroot_seasonal_DJF_MAM_JJAS_ON.png
Processing seasonal climatology for: TVDI_Z
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\TVDI_Z_seasonal_DJF_MAM_JJAS_ON.png
Processing seasonal climatology for: VOD
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VOD_seasonal_DJF_MAM_JJAS_ON.png
Processing seasonal climatology for: VDI
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_seasonal_DJF_MAM_JJAS_ON.png
Processing seasonal climatology for: VDI_Cropland
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\VDI_Cropland_seaso

In [27]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from pathlib import Path
import geopandas as gpd

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
data_dir = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
shp_path = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

# 9 variables to plot
datasets_info = [
    {"file": "ESI_dryz.nc",                             "var": "ESI_dryz",  "title": "ESI"},
    {"file": "SIF_dryz.nc",                             "var": "SIF_dryz",  "title": "SIF"},
    {"file": "SMroot_dryz.nc",                          "var": "SMroot_dryz", "title": "SMroot"},
    {"file": "TVDI_Z_monthly_India_0p05deg_2001-2023.nc","var": "TVDI_Z",   "title": "TVDI_Z"},
    {"file": "VOD_dryz.nc",                             "var": "VOD_dryz",  "title": "VOD"},
    {"file": "VDI_allbiomes.nc",                        "var": "VDI",       "title": "VDI"},
    {"file": "VDI_cropland.nc",                         "var": "VDI",       "title": "VDI_Cropland"},
    {"file": "VDI_forest.nc",                           "var": "VDI",       "title": "VDI_Forest"},
    {"file": "VDI_shrub_grass.nc",                      "var": "VDI",       "title": "VDI_Shrub_Grass"},
]

# ---------------------------------------------------------------------
# Load shapefile
# ---------------------------------------------------------------------
india_gdf = gpd.read_file(shp_path)

# Ensure shapefile is in lat/lon
if india_gdf.crs is not None and india_gdf.crs.to_string() != "EPSG:4326":
    india_gdf = india_gdf.to_crs(epsg=4326)

# ---------------------------------------------------------------------
# Compute JJAS mean for each variable
# ---------------------------------------------------------------------
jjas_means = []

for info in datasets_info:
    print(f"Processing JJAS mean for: {info['title']}")
    ds = xr.open_dataset(data_dir / info["file"])
    da = ds[info["var"]]

    # Select JJAS months: June (6), July (7), August (8), September (9)
    jjas = da.where(da["time"].dt.month.isin([6, 7, 8, 9]), drop=True)

    # Mean over all JJAS months across all years
    jjas_mean = jjas.mean("time", skipna=True)
    jjas_means.append(jjas_mean)

    ds.close()

# ---------------------------------------------------------------------
# Colour settings: –2 to +2, 0 = white
# ---------------------------------------------------------------------
vmin, vmax = -1, 1
norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

colors = [
    "#313695", "#4575b4", "#74add1", "#abd9e9",
    "#ffffff",
    "#fdae61", "#f46d43", "#d73027", "#a50026"
]
cmap = LinearSegmentedColormap.from_list("custom_div", colors)

# ---------------------------------------------------------------------
# Plot 3x3 panel (all variables, JJAS only)
# ---------------------------------------------------------------------
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
axes = axes.ravel()

im = None

for ax, jjas_mean, info in zip(axes, jjas_means, datasets_info):
    # Use each dataset's own grid (handles TVDI_Z with 605x582)
    lats = jjas_mean["lat"].values
    lons = jjas_mean["lon"].values
    lon2d, lat2d = np.meshgrid(lons, lats)

    im = ax.pcolormesh(
        lon2d,
        lat2d,
        jjas_mean.values,
        cmap=cmap,
        norm=norm,
        shading="auto"   # auto chooses 'nearest' for same-shaped X,Y,C
    )

    # Overlay India boundary
    india_gdf.boundary.plot(ax=ax, color="black", linewidth=0.5)

    ax.set_title(info["title"] + " (JJAS)", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

# ---------------------------------------------------------------------
# Shared colorbar on the right
# ---------------------------------------------------------------------
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Standardized anomaly (z-score)")

fig.suptitle("JJAS mean (all variables)", fontsize=14)

# ---------------------------------------------------------------------
# Save figure
# ---------------------------------------------------------------------
out_file = data_dir / "JJAS_3x3_panel_all_variables_v01.png"
fig.savefig(out_file, dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_file)


Processing JJAS mean for: ESI
Processing JJAS mean for: SIF
Processing JJAS mean for: SMroot
Processing JJAS mean for: TVDI_Z
Processing JJAS mean for: VOD
Processing JJAS mean for: VDI
Processing JJAS mean for: VDI_Cropland
Processing JJAS mean for: VDI_Forest
Processing JJAS mean for: VDI_Shrub_Grass
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\JJAS_3x3_panel_all_variables_v01.png


In [29]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from pathlib import Path
import geopandas as gpd

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
data_dir = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
shp_path = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

# 9 variables to plot
datasets_info = [
    {"file": "ESI_dryz.nc",                             "var": "ESI_dryz",  "title": "ESI"},
    {"file": "SIF_dryz.nc",                             "var": "SIF_dryz",  "title": "SIF"},
    {"file": "SMroot_dryz.nc",                          "var": "SMroot_dryz", "title": "SMroot"},
    {"file": "TVDI_Z_monthly_India_0p05deg_2001-2023.nc","var": "TVDI_Z",   "title": "TVDI_Z"},
    {"file": "VOD_dryz.nc",                             "var": "VOD_dryz",  "title": "VOD"},
    {"file": "VDI_allbiomes.nc",                        "var": "VDI",       "title": "VDI"},
    {"file": "VDI_cropland.nc",                         "var": "VDI",       "title": "VDI_Cropland"},
    {"file": "VDI_forest.nc",                           "var": "VDI",       "title": "VDI_Forest"},
    {"file": "VDI_shrub_grass.nc",                      "var": "VDI",       "title": "VDI_Shrub_Grass"},
]

# ---------------------------------------------------------------------
# Load shapefile
# ---------------------------------------------------------------------
india_gdf = gpd.read_file(shp_path)

# Ensure shapefile is in lat/lon
if india_gdf.crs is not None and india_gdf.crs.to_string() != "EPSG:4326":
    india_gdf = india_gdf.to_crs(epsg=4326)

# ---------------------------------------------------------------------
# Compute DJF mean for each variable
# ---------------------------------------------------------------------
djf_means = []

for info in datasets_info:
    print(f"Processing DJF mean for: {info['title']}")
    ds = xr.open_dataset(data_dir / info["file"])
    da = ds[info["var"]]

    # Select DJF months: December (12), January (1), February (2)
    djf = da.where(da["time"].dt.month.isin([12, 1, 2]), drop=True)

    # Mean over all DJF months across all years
    djf_mean = djf.mean("time", skipna=True)
    djf_means.append(djf_mean)

    ds.close()

# ---------------------------------------------------------------------
# Colour settings: –2 to +2, 0 = white
# ---------------------------------------------------------------------
vmin, vmax = -1, 1
norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

colors = [
    "#313695", "#4575b4", "#74add1", "#abd9e9",
    "#ffffff",
    "#fdae61", "#f46d43", "#d73027", "#a50026"
]
cmap = LinearSegmentedColormap.from_list("custom_div", colors)

# ---------------------------------------------------------------------
# Plot 3x3 panel (all variables, DJF only)
# ---------------------------------------------------------------------
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
axes = axes.ravel()

im = None

for ax, djf_mean, info in zip(axes, djf_means, datasets_info):
    # Use each dataset's own grid (handles TVDI_Z with 605x582)
    lats = djf_mean["lat"].values
    lons = djf_mean["lon"].values
    lon2d, lat2d = np.meshgrid(lons, lats)

    im = ax.pcolormesh(
        lon2d,
        lat2d,
        djf_mean.values,
        cmap=cmap,
        norm=norm,
        shading="auto"   # lets Matplotlib choose suitable shading
    )

    # Overlay India boundary
    india_gdf.boundary.plot(ax=ax, color="black", linewidth=0.5)

    ax.set_title(info["title"] + " (DJF)", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

# ---------------------------------------------------------------------
# Shared colorbar on the right
# ---------------------------------------------------------------------
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Standardized anomaly (z-score)")

fig.suptitle("DJF mean (all variables)", fontsize=14)

# ---------------------------------------------------------------------
# Save figure
# ---------------------------------------------------------------------
out_file = data_dir / "DJF_3x3_panel_all_variables.png"
fig.savefig(out_file, dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_file)


Processing DJF mean for: ESI
Processing DJF mean for: SIF
Processing DJF mean for: SMroot
Processing DJF mean for: TVDI_Z
Processing DJF mean for: VOD
Processing DJF mean for: VDI
Processing DJF mean for: VDI_Cropland
Processing DJF mean for: VDI_Forest
Processing DJF mean for: VDI_Shrub_Grass
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\DJF_3x3_panel_all_variables.png


In [31]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
data_dir = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")

datasets_info = [
    {"file": "ESI_dryz.nc",                             "var": "ESI_dryz",  "title": "ESI"},
    {"file": "SIF_dryz.nc",                             "var": "SIF_dryz",  "title": "SIF"},
    {"file": "SMroot_dryz.nc",                          "var": "SMroot_dryz", "title": "SMroot"},
    {"file": "TVDI_Z_monthly_India_0p05deg_2001-2023.nc","var": "TVDI_Z",   "title": "TVDI_Z"},
    {"file": "VOD_dryz.nc",                             "var": "VOD_dryz",  "title": "VOD"},
    {"file": "VDI_allbiomes.nc",                        "var": "VDI",       "title": "VDI"},
    {"file": "VDI_cropland.nc",                         "var": "VDI",       "title": "VDI_Cropland"},
    {"file": "VDI_forest.nc",                           "var": "VDI",       "title": "VDI_Forest"},
    {"file": "VDI_shrub_grass.nc",                      "var": "VDI",       "title": "VDI_Shrub_Grass"},
]

# Seasons definition
season_defs = {
    "Annual": None,             # special handling
    "DJF":    [12, 1, 2],
    "MAM":    [3, 4, 5],
    "JJAS":   [6, 7, 8, 9],
    "ON":     [10, 11],
}

# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------
def timeseries_annual(da):
    """
    Annual mean (calendar year) time series of area mean.
    Returns: years (1D array), values (1D array)
    """
    # group by year, mean over time, then spatial mean
    ann = da.groupby('time.year').mean(dim='time', skipna=True)
    ann = ann.mean(dim=('lat', 'lon'), skipna=True)
    years = ann['year'].values
    values = ann.values
    return years, values

def timeseries_season(da, season_name, months):
    """
    Seasonal mean (MAM, JJAS, ON) time series of area mean.
    For DJF, uses special cross-year handling (Dec belongs to next year).
    Returns: years (1D array), values (1D array)
    """
    t = da['time']
    month = t.dt.month
    year = t.dt.year

    if season_name == "DJF":
        # select only Dec/Jan/Feb
        mask = month.isin(months)
        da_sel = da.where(mask, drop=True)
        month_sel = month.where(mask, drop=True)
        year_sel = year.where(mask, drop=True)

        # Dec -> year+1, Jan/Feb -> same year
        season_year = xr.where(month_sel == 12, year_sel + 1, year_sel)
        season_year = season_year.rename('season_year')

        # group by season_year, mean over time, then spatial mean
        djf = da_sel.groupby(season_year).mean(dim='time', skipna=True)
        djf = djf.mean(dim=('lat', 'lon'), skipna=True)

        years = djf['season_year'].values
        values = djf.values
        return years, values

    else:
        # normal season (all in same calendar year)
        mask = month.isin(months)
        da_sel = da.where(mask, drop=True)

        seas = da_sel.groupby('time.year').mean(dim='time', skipna=True)
        seas = seas.mean(dim=('lat', 'lon'), skipna=True)

        years = seas['year'].values
        values = seas.values
        return years, values

def compute_timeseries_for_season(info, season_name):
    """
    Load dataset, compute area-mean time series for a given season.
    """
    ds = xr.open_dataset(data_dir / info["file"])
    da = ds[info["var"]]

    if season_name == "Annual":
        years, vals = timeseries_annual(da)
    else:
        months = season_defs[season_name]
        years, vals = timeseries_season(da, season_name, months)

    ds.close()
    return years, vals

def make_panel(season_name):
    """
    Make a 3x3 panel of time series for the given season
    (Annual, DJF, MAM, JJAS, ON).
    """
    fig, axes = plt.subplots(3, 3, figsize=(12, 10))
    axes = axes.ravel()

    for ax, info in zip(axes, datasets_info):
        years, vals = compute_timeseries_for_season(info, season_name)

        ax.plot(years, vals, marker='o', linewidth=1.5)
        ax.axhline(0, color='grey', linewidth=0.7, linestyle='--')

        ax.set_title(info["title"], fontsize=10)
        ax.set_xlabel("Year")
        ax.set_ylabel("Index")

        # make x-ticks a bit sparser if many years
        if len(years) > 15:
            step = max(1, len(years) // 10)
            ax.set_xticks(years[::step])

        ax.tick_params(axis='x', rotation=45)

    fig.suptitle(f"{season_name} area-mean time series (all variables)", fontsize=14)
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])

    out_file = data_dir / f"{season_name}_timeseries_3x3_panel.png"
    fig.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_file}")

# ---------------------------------------------------------------------
# Generate all panels
# ---------------------------------------------------------------------
for season_name in ["Annual", "DJF", "MAM", "JJAS", "ON"]:
    print(f"=== {season_name} ===")
    make_panel(season_name)


=== Annual ===
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\Annual_timeseries_3x3_panel.png
=== DJF ===
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\DJF_timeseries_3x3_panel.png
=== MAM ===
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\MAM_timeseries_3x3_panel.png
=== JJAS ===
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\JJAS_timeseries_3x3_panel.png
=== ON ===
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\ON_timeseries_3x3_panel.png


In [33]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from pathlib import Path
import geopandas as gpd

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
data_dir = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
shp_path = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

# Variables / files
datasets_info = [
    {"file": "ESI_dryz.nc",                             "var": "ESI_dryz",  "title": "ESI"},
    {"file": "SIF_dryz.nc",                             "var": "SIF_dryz",  "title": "SIF"},
    {"file": "SMroot_dryz.nc",                          "var": "SMroot_dryz", "title": "SMroot"},
    {"file": "TVDI_Z_monthly_India_0p05deg_2001-2023.nc","var": "TVDI_Z",   "title": "TVDI_Z"},
    {"file": "VOD_dryz.nc",                             "var": "VOD_dryz",  "title": "VOD"},
    {"file": "VDI_allbiomes.nc",                        "var": "VDI",       "title": "VDI"},
    {"file": "VDI_cropland.nc",                         "var": "VDI",       "title": "VDI_Cropland"},
    {"file": "VDI_forest.nc",                           "var": "VDI",       "title": "VDI_Forest"},
    {"file": "VDI_shrub_grass.nc",                      "var": "VDI",       "title": "VDI_Shrub_Grass"},
]

# Years and seasons to plot
years_to_plot = [2002, 2005, 2009, 2015, 2019]
seasons_to_plot = ["Annual", "JJAS", "DJF"]

season_months = {
    "JJAS": [6, 7, 8, 9],
    "DJF":  [12, 1, 2],
}

# ---------------------------------------------------------------------
# Load shapefile
# ---------------------------------------------------------------------
india_gdf = gpd.read_file(shp_path)

# Ensure shapefile is in lat/lon
if india_gdf.crs is not None and india_gdf.crs.to_string() != "EPSG:4326":
    india_gdf = india_gdf.to_crs(epsg=4326)

# ---------------------------------------------------------------------
# Colour settings: –2 to +2, 0 = white
# ---------------------------------------------------------------------
vmin, vmax = -2, 2
norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

colors = [
    "#313695", "#4575b4", "#74add1", "#abd9e9",
    "#ffffff",
    "#fdae61", "#f46d43", "#d73027", "#a50026"
]
cmap = LinearSegmentedColormap.from_list("custom_div", colors)

# ---------------------------------------------------------------------
# Helper: select data for given year & season and compute mean
# ---------------------------------------------------------------------
def select_season_year_mean(da, season, year):
    """
    For a DataArray da(time, lat, lon), compute mean over time
    for a given season and year.

    - Annual: all months in that calendar year
    - JJAS:   months [6,7,8,9] in that year
    - DJF:    Dec (previous year) + Jan, Feb (given year),
              season labeled by the year of Jan/Feb.
    """
    t = da["time"]
    month = t.dt.month
    yr = t.dt.year

    if season == "Annual":
        mask = (yr == year)
        da_sel = da.where(mask, drop=True)

    elif season == "JJAS":
        mask = (yr == year) & (month.isin(season_months["JJAS"]))
        da_sel = da.where(mask, drop=True)

    elif season == "DJF":
        # mask DJF months only
        mask_djf = month.isin(season_months["DJF"])
        da_djf = da.where(mask_djf, drop=True)
        month_djf = month.where(mask_djf, drop=True)
        year_djf = yr.where(mask_djf, drop=True)

        # assign "season year": Dec -> year+1, Jan/Feb -> same year
        season_year = xr.where(month_djf == 12, year_djf + 1, year_djf)
        # select for the requested DJF season-year
        mask_season = (season_year == year)
        da_sel = da_djf.where(mask_season, drop=True)

    else:
        raise ValueError(f"Unknown season: {season}")

    if da_sel.sizes.get("time", 0) == 0:
        # no data for this season/year (shouldn't happen here, but safe)
        return None

    return da_sel.mean(dim="time", skipna=True)

# ---------------------------------------------------------------------
# Main: loop over seasons and years and plot 3x3 maps
# ---------------------------------------------------------------------
for season in seasons_to_plot:
    for year in years_to_plot:
        print(f"Processing {season} {year}...")

        # Compute mean maps for all variables for this season/year
        mean_fields = []
        for info in datasets_info:
            ds = xr.open_dataset(data_dir / info["file"])
            da = ds[info["var"]]

            mean_da = select_season_year_mean(da, season, year)
            ds.close()

            if mean_da is None:
                # if missing, create a NaN array on same lat/lon grid
                mean_da = da.isel(time=0, drop=True) * np.nan

            mean_fields.append(mean_da)

        # -------------------------------------------------------------
        # 3x3 panel
        # -------------------------------------------------------------
        fig, axes = plt.subplots(3, 3, figsize=(12, 10))
        axes = axes.ravel()

        im = None

        for ax, mean_da, info in zip(axes, mean_fields, datasets_info):
            lats = mean_da["lat"].values
            lons = mean_da["lon"].values
            lon2d, lat2d = np.meshgrid(lons, lats)

            im = ax.pcolormesh(
                lon2d,
                lat2d,
                mean_da.values,
                cmap=cmap,
                norm=norm,
                shading="auto",
            )

            india_gdf.boundary.plot(ax=ax, color="black", linewidth=0.5)

            ax.set_title(f"{info['title']}", fontsize=10)
            ax.set_xticks([])
            ax.set_yticks([])

        # Shared colorbar
        fig.subplots_adjust(right=0.88)
        cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
        cbar = fig.colorbar(im, cax=cbar_ax)
        cbar.set_label("Standardized anomaly (z-score)")

        fig.suptitle(f"{season} mean, year {year} (all variables)", fontsize=14)

        out_file = data_dir / f"{season}_{year}_3x3_panel_all_variables.png"
        fig.savefig(out_file, dpi=300, bbox_inches="tight")
        plt.close(fig)

        print(f"  Saved: {out_file}")


Processing Annual 2002...
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\Annual_2002_3x3_panel_all_variables.png
Processing Annual 2005...
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\Annual_2005_3x3_panel_all_variables.png
Processing Annual 2009...
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\Annual_2009_3x3_panel_all_variables.png
Processing Annual 2015...
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\Annual_2015_3x3_panel_all_variables.png
Processing Annual 2019...
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\Annual_2019_3x3_panel_all_variables.png
Processing JJAS 2002...
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\JJAS_2002_3x3_panel_all_variables.png
Processing JJAS 2005...
  Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\JJAS_2005_3x3_panel_all_variables.png
Processing JJAS 2009...
  Saved: C:\Drought\Regridding and data clippi

In [39]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from pathlib import Path
import geopandas as gpd

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
data_dir = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
shp_path = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

# Variables to include (exclude VDI_cropland / VDI_forest / VDI_shrub_grass)
vars_info = [
    {"file": "ESI_dryz.nc",                             "var": "ESI_dryz",  "name": "ESI"},
    {"file": "SIF_dryz.nc",                             "var": "SIF_dryz",  "name": "SIF"},
    {"file": "SMroot_dryz.nc",                          "var": "SMroot_dryz", "name": "SMroot"},
    {"file": "TVDI_Z_monthly_India_0p05deg_2001-2023.nc","var": "TVDI_Z",   "name": "TVDI_Z"},
    {"file": "VOD_dryz.nc",                             "var": "VOD_dryz",  "name": "VOD"},
    {"file": "VDI_allbiomes.nc",                        "var": "VDI",       "name": "VDI"},
]

# ---------------------------------------------------------------------
# Load shapefile
# ---------------------------------------------------------------------
india_gdf = gpd.read_file(shp_path)

# Ensure shapefile is in lat/lon
if india_gdf.crs is not None and india_gdf.crs.to_string() != "EPSG:4326":
    india_gdf = india_gdf.to_crs(epsg=4326)

# ---------------------------------------------------------------------
# Reference grid from VDI_allbiomes (611 x 587)
# ---------------------------------------------------------------------
ref_ds = xr.open_dataset(data_dir / "VDI_allbiomes.nc")
ref_lat = ref_ds["lat"]
ref_lon = ref_ds["lon"]
ref_ds.close()

# ---------------------------------------------------------------------
# Prepare each variable:
#   - regrid to VDI grid
#   - standardise time to integer month_id = year*100 + month
# ---------------------------------------------------------------------
prepared_das = []
var_names = []

for info in vars_info:
    print(f"Loading and preparing: {info['name']}")

    ds = xr.open_dataset(data_dir / info["file"])
    da = ds[info["var"]]

    # Regrid to reference grid
    da_rg = da.interp(lat=ref_lat, lon=ref_lon)

    # Build integer month_id = YYYYMM
    year = da_rg["time"].dt.year
    month = da_rg["time"].dt.month
    month_id = (year * 100 + month).rename("month_id")

    # Group by month_id so that each calendar month is one "time" step
    da_mon = da_rg.groupby(month_id).mean("time", skipna=True)

    # month_id is now a coordinate; rename it to "time" dimension
    da_mon = da_mon.rename({"month_id": "time"})

    prepared_das.append(da_mon)
    var_names.append(info["name"])

    ds.close()

# ---------------------------------------------------------------------
# Align all variables on common "time" (intersection of month_id)
# ---------------------------------------------------------------------
print("Aligning time across all variables...")
aligned_das = xr.align(*prepared_das, join="inner")  # list of DataArrays

# ---------------------------------------------------------------------
# Compute pairwise temporal correlation maps (over 'time')
# ---------------------------------------------------------------------
corr_maps = []
corr_labels = []

n = len(aligned_das)
for i in range(n):
    for j in range(i + 1, n):
        da_i = aligned_das[i]
        da_j = aligned_das[j]
        label = f"{var_names[i]} vs {var_names[j]}"

        print(f"Computing correlation: {label}")

        # Pearson correlation over time dimension -> (lat, lon)
        corr_ij = xr.corr(da_i, da_j, dim="time")

        corr_maps.append(corr_ij)
        corr_labels.append(label)

# ---------------------------------------------------------------------
# Plot correlation maps in a single 3x5 panel
# ---------------------------------------------------------------------
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.ravel()

# Colour settings: correlation from -1 to 1, 0 = white
vmin, vmax = -1.0, 1.0
norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

colors = [
    "#313695", "#4575b4", "#74add1", "#abd9e9",
    "#ffffff",
    "#fdae61", "#f46d43", "#d73027", "#a50026"
]
cmap = LinearSegmentedColormap.from_list("corr_div", colors)

# Prepare lon/lat grid from reference
lats = ref_lat.values
lons = ref_lon.values
lon2d, lat2d = np.meshgrid(lons, lats)

im = None

for ax, corr_da, label in zip(axes, corr_maps, corr_labels):
    im = ax.pcolormesh(
        lon2d,
        lat2d,
        corr_da.values,
        cmap=cmap,
        norm=norm,
        shading="auto",
    )

    # Overlay India boundary
    india_gdf.boundary.plot(ax=ax, color="black", linewidth=0.5)

    ax.set_title(label, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

# Hide any unused axes (there should be none with 15 maps)
for k in range(len(corr_maps), len(axes)):
    axes[k].set_visible(False)

# Shared colorbar on the right
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Pearson correlation (r)")

fig.suptitle("Spatial correlation between drought indices (monthly, overlapping period)", fontsize=14)

# Save figure
out_file = data_dir / "correlation_all_variables_3x5_panel.png"
fig.savefig(out_file, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"Saved correlation figure to: {out_file}")


Loading and preparing: ESI
Loading and preparing: SIF
Loading and preparing: SMroot
Loading and preparing: TVDI_Z
Loading and preparing: VOD
Loading and preparing: VDI
Aligning time across all variables...
Computing correlation: ESI vs SIF
Computing correlation: ESI vs SMroot
Computing correlation: ESI vs TVDI_Z
Computing correlation: ESI vs VOD
Computing correlation: ESI vs VDI
Computing correlation: SIF vs SMroot
Computing correlation: SIF vs TVDI_Z
Computing correlation: SIF vs VOD
Computing correlation: SIF vs VDI
Computing correlation: SMroot vs TVDI_Z
Computing correlation: SMroot vs VOD
Computing correlation: SMroot vs VDI
Computing correlation: TVDI_Z vs VOD
Computing correlation: TVDI_Z vs VDI
Computing correlation: VOD vs VDI
Saved correlation figure to: C:\Drought\Regridding and data clipping\Final Analysis\VDI\correlation_all_variables_3x5_panel.png


In [41]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from pathlib import Path
import geopandas as gpd

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
data_dir = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
shp_path = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

# Variables to include (exclude VDI_cropland / VDI_forest / VDI_shrub_grass)
vars_info = [
    {"file": "ESI_dryz.nc",                             "var": "ESI_dryz",  "name": "ESI"},
    {"file": "SIF_dryz.nc",                             "var": "SIF_dryz",  "name": "SIF"},
    {"file": "SMroot_dryz.nc",                          "var": "SMroot_dryz", "name": "SMroot"},
    {"file": "TVDI_Z_monthly_India_0p05deg_2001-2023.nc","var": "TVDI_Z",   "name": "TVDI_Z"},
    {"file": "VOD_dryz.nc",                             "var": "VOD_dryz",  "name": "VOD"},
    {"file": "VDI_allbiomes.nc",                        "var": "VDI",       "name": "VDI"},
]

# Lags (in months) to compute
lags = [1, 2, 3]

# ---------------------------------------------------------------------
# Load shapefile
# ---------------------------------------------------------------------
india_gdf = gpd.read_file(shp_path)

# Ensure shapefile is in lat/lon
if india_gdf.crs is not None and india_gdf.crs.to_string() != "EPSG:4326":
    india_gdf = india_gdf.to_crs(epsg=4326)

# ---------------------------------------------------------------------
# Reference grid from VDI_allbiomes (611 x 587)
# ---------------------------------------------------------------------
ref_ds = xr.open_dataset(data_dir / "VDI_allbiomes.nc")
ref_lat = ref_ds["lat"]
ref_lon = ref_ds["lon"]
ref_ds.close()

# ---------------------------------------------------------------------
# Prepare each variable:
#   - regrid to VDI grid
#   - standardise time to integer month_id = year*100 + month
# ---------------------------------------------------------------------
prepared_das = []
var_names = []

for info in vars_info:
    print(f"Loading and preparing: {info['name']}")

    ds = xr.open_dataset(data_dir / info["file"])
    da = ds[info["var"]]

    # Regrid to reference grid
    da_rg = da.interp(lat=ref_lat, lon=ref_lon)

    # Build integer month_id = YYYYMM
    year = da_rg["time"].dt.year
    month = da_rg["time"].dt.month
    month_id = (year * 100 + month).rename("month_id")

    # Group by month_id so that each calendar month is one "time" step
    da_mon = da_rg.groupby(month_id).mean("time", skipna=True)

    # month_id is now a coordinate; rename it to "time" dimension
    da_mon = da_mon.rename({"month_id": "time"})

    prepared_das.append(da_mon)
    var_names.append(info["name"])

    ds.close()

# ---------------------------------------------------------------------
# Align all variables on common "time" (intersection of month_id)
# ---------------------------------------------------------------------
print("Aligning time across all variables...")
aligned_das = xr.align(*prepared_das, join="inner")  # list of DataArrays

# Pre-build lon/lat mesh
lats = ref_lat.values
lons = ref_lon.values
lon2d, lat2d = np.meshgrid(lons, lats)

# Colour settings for correlation: -1 to 1, 0 = white
vmin, vmax = -1.0, 1.0
norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
colors = [
    "#313695", "#4575b4", "#74add1", "#abd9e9",
    "#ffffff",
    "#fdae61", "#f46d43", "#d73027", "#a50026"
]
cmap = LinearSegmentedColormap.from_list("corr_div", colors)

# ---------------------------------------------------------------------
# Loop over lags and compute correlation maps
# ---------------------------------------------------------------------
for lag in lags:
    print(f"\n=== Computing lag-{lag} correlations (X(t) vs Y(t+{lag})) ===")

    corr_maps = []
    corr_labels = []

    n = len(aligned_das)
    for i in range(n):
        for j in range(i + 1, n):
            da_i = aligned_das[i]
            da_j = aligned_das[j]

            # Shift the second variable forward by 'lag' months
            # so we correlate X(t) with Y(t+lag).
            da_j_lag = da_j.shift(time=-lag)

            label = f"{var_names[i]} vs {var_names[j]} (lag {lag})"
            print(f"  Pair: {label}")

            # xr.corr will ignore time steps where either is NaN (from shifting)
            corr_ij = xr.corr(da_i, da_j_lag, dim="time")

            corr_maps.append(corr_ij)
            corr_labels.append(label)

    # -------------------------------------------------------------
    # Plot correlation maps in a single 3x5 panel for this lag
    # -------------------------------------------------------------
    fig, axes = plt.subplots(3, 5, figsize=(18, 10))
    axes = axes.ravel()

    im = None
    for ax, corr_da, label in zip(axes, corr_maps, corr_labels):
        im = ax.pcolormesh(
            lon2d,
            lat2d,
            corr_da.values,
            cmap=cmap,
            norm=norm,
            shading="auto",
        )

        # Overlay India boundary
        india_gdf.boundary.plot(ax=ax, color="black", linewidth=0.5)

        ax.set_title(label, fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

    # Hide any unused axes (should be none with 15 maps)
    for k in range(len(corr_maps), len(axes)):
        axes[k].set_visible(False)

    # Shared colorbar on the right
    fig.subplots_adjust(right=0.9)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label("Pearson correlation (r)")

    fig.suptitle(f"Spatial lag-{lag} correlation between drought indices (monthly, overlapping period)", fontsize=14)

    # Save figure
    out_file = data_dir / f"correlation_lag{lag}_3x5_panel.png"
    fig.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved lag-{lag} correlation figure to: {out_file}")


Loading and preparing: ESI
Loading and preparing: SIF
Loading and preparing: SMroot
Loading and preparing: TVDI_Z
Loading and preparing: VOD
Loading and preparing: VDI
Aligning time across all variables...

=== Computing lag-1 correlations (X(t) vs Y(t+1)) ===
  Pair: ESI vs SIF (lag 1)
  Pair: ESI vs SMroot (lag 1)
  Pair: ESI vs TVDI_Z (lag 1)
  Pair: ESI vs VOD (lag 1)
  Pair: ESI vs VDI (lag 1)
  Pair: SIF vs SMroot (lag 1)
  Pair: SIF vs TVDI_Z (lag 1)
  Pair: SIF vs VOD (lag 1)
  Pair: SIF vs VDI (lag 1)
  Pair: SMroot vs TVDI_Z (lag 1)
  Pair: SMroot vs VOD (lag 1)
  Pair: SMroot vs VDI (lag 1)
  Pair: TVDI_Z vs VOD (lag 1)
  Pair: TVDI_Z vs VDI (lag 1)
  Pair: VOD vs VDI (lag 1)
Saved lag-1 correlation figure to: C:\Drought\Regridding and data clipping\Final Analysis\VDI\correlation_lag1_3x5_panel.png

=== Computing lag-2 correlations (X(t) vs Y(t+2)) ===
  Pair: ESI vs SIF (lag 2)
  Pair: ESI vs SMroot (lag 2)
  Pair: ESI vs TVDI_Z (lag 2)
  Pair: ESI vs VOD (lag 2)
  Pair: ES

In [43]:
import xarray as xr
from pathlib import Path

# ----------------------------
# File paths
# ----------------------------
spi_path   = Path(r"C:\Drought\SPI\SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc")
pdsi_path  = Path(r"C:\Drought\Regridding and data clipping\PDSI\TerraClimate_PDSI_0p05deg_INDIA.nc")
grace_path = Path(r"C:\Drought\Regridding and data clipping\GRACE\GRACE_JPL_mascon_India_0p05deg_2003-2023.nc")

# ----------------------------
# Helper: Print dataset summary
# ----------------------------
def describe_dataset(ds, name):
    print("="*80)
    print(f"File: {name}")
    print("="*80)
    print("\n--- xarray dataset summary ---")
    print(ds)
    print("\n--- Variable details ---\n")
    for var in ds.data_vars:
        da = ds[var]
        print(f"Variable name: {var}")
        print(f"  dims:  {da.dims}")
        print(f"  shape: {da.shape}")
        print(f"  attributes: {da.attrs}")
        print("\n")
    print("="*80 + "\n")


# ----------------------------
# Load and describe SPI
# ----------------------------
print("Loading SPI dataset...")
spi_ds = xr.open_dataset(spi_path)
describe_dataset(spi_ds, "SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc")

# ----------------------------
# Load and describe PDSI
# ----------------------------
print("Loading PDSI dataset...")
pdsi_ds = xr.open_dataset(pdsi_path)
describe_dataset(pdsi_ds, "TerraClimate_PDSI_0p05deg_INDIA.nc")

# ----------------------------
# Load and describe GRACE data
# ----------------------------
print("Loading GRACE dataset...")
grace_ds = xr.open_dataset(grace_path)
describe_dataset(grace_ds, "GRACE_JPL_mascon_India_0p05deg_2003-2023.nc")


Loading SPI dataset...
File: SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc

--- xarray dataset summary ---
<xarray.Dataset> Size: 2GB
Dimensions:  (lat: 605, lon: 582, time: 288)
Coordinates:
  * lat      (lat) float32 2kB 6.775 6.825 6.875 6.925 ... 36.87 36.92 36.97
  * lon      (lon) float32 2kB 68.38 68.43 68.47 68.52 ... 97.33 97.38 97.42
  * time     (time) datetime64[ns] 2kB 2000-01-01 2000-02-01 ... 2023-12-01
    month    (time) int64 2kB ...
Data variables:
    SPI1     (lat, lon, time) float32 406MB ...
    SPI3     (lat, lon, time) float32 406MB ...
    SPI6     (lat, lon, time) float32 406MB ...
    SPI9     (lat, lon, time) float32 406MB ...
    SPI12    (lat, lon, time) float32 406MB ...

--- Variable details ---

Variable name: SPI1
  dims:  ('lat', 'lon', 'time')
  shape: (605, 582, 288)
  attributes: {'units': 'mm/month', 'standard_name': 'convective precipitation rate', 'long_name': 'Standardized Precipitation Index (1-month)', 'time_step': 'month', 'geostatial_lat_mi

In [45]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from pathlib import Path
import geopandas as gpd

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
vdi_dir   = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
spi_dir   = Path(r"C:\Drought\SPI")
pdsi_dir  = Path(r"C:\Drought\Regridding and data clipping\PDSI")
grace_dir = Path(r"C:\Drought\Regridding and data clipping\GRACE")
shp_path  = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

# ---------------------------------------------------------------------
# Variables to include in the 5x5 panel
# ---------------------------------------------------------------------
# Note: for GRACE I use GAD_msc_Lmax180 here. Change "var" to GAA/GAB/GAC if needed.
datasets_info = [
    {
        "name": "VDI",
        "file": vdi_dir / "VDI_allbiomes.nc",
        "var": "VDI",
    },
    {
        "name": "PDSI",
        "file": pdsi_dir / "TerraClimate_PDSI_0p05deg_INDIA.nc",
        "var": "PDSI",
    },
    {
        "name": "GRACE",
        "file": grace_dir / "GRACE_JPL_mascon_India_0p05deg_2003-2023.nc",
        "var": "GAD_msc_Lmax180",
    },
    {
        "name": "SPI3",
        "file": spi_dir / "SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc",
        "var": "SPI3",
    },
    {
        "name": "SPI6",
        "file": spi_dir / "SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc",
        "var": "SPI6",
    },
]

seasons = ["Annual", "DJF", "MAM", "JJAS", "ON"]
season_months = {
    "DJF":  [12, 1, 2],
    "MAM":  [3, 4, 5],
    "JJAS": [6, 7, 8, 9],
    "ON":   [10, 11],
}

# ---------------------------------------------------------------------
# Load shapefile (India boundary)
# ---------------------------------------------------------------------
india_gdf = gpd.read_file(shp_path)
if india_gdf.crs is not None and india_gdf.crs.to_string() != "EPSG:4326":
    india_gdf = india_gdf.to_crs(epsg=4326)

# ---------------------------------------------------------------------
# Reference grid from VDI_allbiomes (target grid for all maps)
# ---------------------------------------------------------------------
ref_ds = xr.open_dataset(vdi_dir / "VDI_allbiomes.nc")
ref_lat = ref_ds["lat"]
ref_lon = ref_ds["lon"]
ref_ds.close()

# Precompute lon/lat mesh on reference grid
lats_ref = ref_lat.values
lons_ref = ref_lon.values
lon2d_ref, lat2d_ref = np.meshgrid(lons_ref, lats_ref)

# ---------------------------------------------------------------------
# Helper: compute annual and seasonal climatology
# ---------------------------------------------------------------------
def compute_climatologies(da):
    """
    Given a DataArray da(time, lat, lon) on the reference grid,
    compute climatological means for:
    - Annual over all time
    - DJF, MAM, JJAS, ON over all years
    Returns dict: season_name -> DataArray(lat, lon)
    """
    clim = {}

    # Annual: mean over all time
    clim["Annual"] = da.mean("time", skipna=True)

    # Monthly index
    months = da["time"].dt.month

    # Pure seasonal means (do not separate by year, just all months matching)
    for s in ["DJF", "MAM", "JJAS", "ON"]:
        mask = months.isin(season_months[s])
        da_sel = da.where(mask, drop=True)
        clim[s] = da_sel.mean("time", skipna=True)

    return clim

# ---------------------------------------------------------------------
# Compute climatologies for all variables on VDI grid
# ---------------------------------------------------------------------
clim_all = {}  # clim_all[var_name][season] = DataArray(lat, lon)

for info in datasets_info:
    print(f"Processing variable: {info['name']}")

    ds = xr.open_dataset(info["file"])
    da = ds[info["var"]]

    # Ensure dimension order is (time, lat, lon)
    dim_order = list(da.dims)
    if dim_order[0] != "time":
        # try to transpose if time is present
        if "time" in dim_order and "lat" in dim_order and "lon" in dim_order:
            da = da.transpose("time", "lat", "lon")
        else:
            raise ValueError(f"Unexpected dims for {info['name']}: {da.dims}")

    # Regrid to VDI grid (if needed)
    da_rg = da.interp(lat=ref_lat, lon=ref_lon)

    # Compute annual + seasonal climatologies
    clim = compute_climatologies(da_rg)

    clim_all[info["name"]] = clim

    ds.close()

# ---------------------------------------------------------------------
# Colour settings: –2 to +2, 0 = white
# ---------------------------------------------------------------------
vmin, vmax = -2, 2
norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

colors = [
    "#313695", "#4575b4", "#74add1", "#abd9e9",
    "#ffffff",
    "#fdae61", "#f46d43", "#d73027", "#a50026"
]
cmap = LinearSegmentedColormap.from_list("custom_div", colors)

# ---------------------------------------------------------------------
# Plot 5x5 panel: rows = variables, columns = seasons
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    nrows=len(datasets_info),
    ncols=len(seasons),
    figsize=(16, 14)
)

for i, info in enumerate(datasets_info):
    var_name = info["name"]
    for j, season in enumerate(seasons):
        ax = axes[i, j]

        da_plot = clim_all[var_name][season]

        im = ax.pcolormesh(
            lon2d_ref,
            lat2d_ref,
            da_plot.values,
            cmap=cmap,
            norm=norm,
            shading="auto",
        )

        # Overlay India boundary
        india_gdf.boundary.plot(ax=ax, color="black", linewidth=0.5)

        # Titles: top row -> season names
        if i == 0:
            ax.set_title(season, fontsize=11)

        # Row labels: left column -> variable names
        if j == 0:
            ax.set_ylabel(var_name, fontsize=11)

        ax.set_xticks([])
        ax.set_yticks([])

# Shared colorbar on the right
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Standardized / drought index units")

fig.suptitle("Annual and seasonal climatologies (VDI, PDSI, GRACE, SPI3, SPI6)", fontsize=16)

# Save figure (in VDI directory)
out_file = vdi_dir / "Annual_DJF_MAM_JJAS_ON_VDI_PDSI_GRACE_SPI3_SPI6_5x5_panel.png"
fig.savefig(out_file, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"Saved 5x5 panel figure to: {out_file}")


Processing variable: VDI
Processing variable: PDSI
Processing variable: GRACE
Processing variable: SPI3
Processing variable: SPI6
Saved 5x5 panel figure to: C:\Drought\Regridding and data clipping\Final Analysis\VDI\Annual_DJF_MAM_JJAS_ON_VDI_PDSI_GRACE_SPI3_SPI6_5x5_panel.png


In [47]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from pathlib import Path
import geopandas as gpd

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
vdi_dir   = Path(r"C:\Drought\Regridding and data clipping\Final Analysis\VDI")
spi_dir   = Path(r"C:\Drought\SPI")
pdsi_dir  = Path(r"C:\Drought\Regridding and data clipping\PDSI")
grace_dir = Path(r"C:\Drought\Regridding and data clipping\GRACE")
shp_path  = Path(r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp")

# ---------------------------------------------------------------------
# Variables and years
# ---------------------------------------------------------------------
datasets_info = [
    {
        "name": "VDI",
        "file": vdi_dir / "VDI_allbiomes.nc",
        "var": "VDI",
    },
    {
        "name": "PDSI",
        "file": pdsi_dir / "TerraClimate_PDSI_0p05deg_INDIA.nc",
        "var": "PDSI",
    },
    {
        "name": "GRACE",
        "file": grace_dir / "GRACE_JPL_mascon_India_0p05deg_2003-2023.nc",
        "var": "GAD_msc_Lmax180",  # change to GAA/GAB/GAC if you prefer
    },
    {
        "name": "SPI3",
        "file": spi_dir / "SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc",
        "var": "SPI3",
    },
    {
        "name": "SPI6",
        "file": spi_dir / "SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc",
        "var": "SPI6",
    },
]

years = [2002, 2005, 2009, 2015, 2019]
seasons = ["Annual", "JJAS", "DJF"]
season_months = {
    "JJAS": [6, 7, 8, 9],
    "DJF":  [12, 1, 2],
}

# ---------------------------------------------------------------------
# Load shapefile
# ---------------------------------------------------------------------
india_gdf = gpd.read_file(shp_path)
if india_gdf.crs is not None and india_gdf.crs.to_string() != "EPSG:4326":
    india_gdf = india_gdf.to_crs(epsg=4326)

# ---------------------------------------------------------------------
# Reference grid from VDI_allbiomes
# ---------------------------------------------------------------------
ref_ds = xr.open_dataset(vdi_dir / "VDI_allbiomes.nc")
ref_lat = ref_ds["lat"]
ref_lon = ref_ds["lon"]
ref_ds.close()

lats_ref = ref_lat.values
lons_ref = ref_lon.values
lon2d_ref, lat2d_ref = np.meshgrid(lons_ref, lats_ref)

# ---------------------------------------------------------------------
# Helper: select data for given year & season and compute mean
# ---------------------------------------------------------------------
def select_season_year_mean(da, season, year):
    """
    da: DataArray(time, lat, lon)
    season: 'Annual', 'JJAS', or 'DJF'
    year: int
    returns: DataArray(lat, lon) or None if no data
    """
    t = da["time"]
    month = t.dt.month
    yr = t.dt.year

    if season == "Annual":
        mask = (yr == year)
        da_sel = da.where(mask, drop=True)

    elif season == "JJAS":
        mask = (yr == year) & month.isin(season_months["JJAS"])
        da_sel = da.where(mask, drop=True)

    elif season == "DJF":
        # select only Dec/Jan/Feb
        mask_djf = month.isin(season_months["DJF"])
        da_djf = da.where(mask_djf, drop=True)
        month_djf = month.where(mask_djf, drop=True)
        year_djf = yr.where(mask_djf, drop=True)

        # Dec counts for next year; Jan/Feb for same year
        season_year = xr.where(month_djf == 12, year_djf + 1, year_djf)

        # select target DJF season
        mask_season = (season_year == year)
        da_sel = da_djf.where(mask_season, drop=True)

    else:
        raise ValueError(f"Unknown season: {season}")

    if da_sel.sizes.get("time", 0) == 0:
        return None

    return da_sel.mean("time", skipna=True)

# ---------------------------------------------------------------------
# Colour settings (same as before)
# ---------------------------------------------------------------------
vmin, vmax = -2, 2
norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

colors = [
    "#313695", "#4575b4", "#74add1", "#abd9e9",
    "#ffffff",
    "#fdae61", "#f46d43", "#d73027", "#a50026"
]
cmap = LinearSegmentedColormap.from_list("custom_div", colors)

# ---------------------------------------------------------------------
# Main loop over seasons: one 5x5 panel per season
# ---------------------------------------------------------------------
for season in seasons:
    print(f"\n=== Building 5x5 panel for {season} ===")

    # mean_fields[var_name][year] = DataArray(lat, lon)
    mean_fields = {info["name"]: {} for info in datasets_info}

    # Compute seasonal/annual means for each variable & year
    for info in datasets_info:
        print(f"  Processing variable: {info['name']}")

        ds = xr.open_dataset(info["file"])
        da = ds[info["var"]]

        # Ensure order (time, lat, lon)
        dims = list(da.dims)
        if set(["time", "lat", "lon"]).issubset(dims):
            # transpose only if necessary
            if dims != ["time", "lat", "lon"]:
                da = da.transpose("time", "lat", "lon")
        else:
            raise ValueError(f"Unexpected dims for {info['name']}: {da.dims}")

        # Regrid to reference grid
        da_rg = da.interp(lat=ref_lat, lon=ref_lon)

        for y in years:
            mean_da = select_season_year_mean(da_rg, season, y)

            if mean_da is None:
                # create NaN field on reference grid if no data
                empty = xr.DataArray(
                    np.full((len(ref_lat), len(ref_lon)), np.nan),
                    coords={"lat": ref_lat, "lon": ref_lon},
                    dims=("lat", "lon"),
                )
                mean_fields[info["name"]][y] = empty
            else:
                mean_fields[info["name"]][y] = mean_da

        ds.close()

    # -------------------------------------------------------------
    # Plotting: rows = variables, cols = years
    # -------------------------------------------------------------
    fig, axes = plt.subplots(
        nrows=len(datasets_info),
        ncols=len(years),
        figsize=(16, 14)
    )

    for i, info in enumerate(datasets_info):
        var_name = info["name"]
        for j, y in enumerate(years):
            ax = axes[i, j]

            da_plot = mean_fields[var_name][y]

            im = ax.pcolormesh(
                lon2d_ref,
                lat2d_ref,
                da_plot.values,
                cmap=cmap,
                norm=norm,
                shading="auto",
            )

            # Overlay India boundary
            india_gdf.boundary.plot(ax=ax, color="black", linewidth=0.5)

            # Column titles: years on top row
            if i == 0:
                ax.set_title(str(y), fontsize=11)

            # Row labels: variable name on left
            if j == 0:
                ax.set_ylabel(var_name, fontsize=11)

            ax.set_xticks([])
            ax.set_yticks([])

    # Shared colorbar
    fig.subplots_adjust(right=0.88)
    cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label("Index value (standardized or native units)")

    fig.suptitle(f"{season} mean – years {years} (VDI, PDSI, GRACE, SPI3, SPI6)", fontsize=16)

    # Save
    out_file = vdi_dir / f"{season}_years_5x5_panel_VDI_PDSI_GRACE_SPI3_SPI6.png"
    fig.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {out_file}")



=== Building 5x5 panel for Annual ===
  Processing variable: VDI
  Processing variable: PDSI
  Processing variable: GRACE
  Processing variable: SPI3
  Processing variable: SPI6
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\Annual_years_5x5_panel_VDI_PDSI_GRACE_SPI3_SPI6.png

=== Building 5x5 panel for JJAS ===
  Processing variable: VDI
  Processing variable: PDSI
  Processing variable: GRACE
  Processing variable: SPI3
  Processing variable: SPI6
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\JJAS_years_5x5_panel_VDI_PDSI_GRACE_SPI3_SPI6.png

=== Building 5x5 panel for DJF ===
  Processing variable: VDI
  Processing variable: PDSI
  Processing variable: GRACE
  Processing variable: SPI3
  Processing variable: SPI6
Saved: C:\Drought\Regridding and data clipping\Final Analysis\VDI\DJF_years_5x5_panel_VDI_PDSI_GRACE_SPI3_SPI6.png
